In [1]:
import sys
sys.path.append('../')

from scripts.algorithms import *

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


## Read and preprocess the data

In [2]:

############
dtx = pd.read_csv("../data/student_math.csv")

print("Original number of rows: ", len(dtx))
print(" ")

dt = dtx.drop_duplicates().reset_index(drop=True)

############
target_column = "pass"

X = dt.drop([target_column], axis=1).values
y = dt[target_column].values
y = np.where(y == 1, 1, -1) 


scaler = StandardScaler()
X = scaler.fit_transform(X)


Original number of rows:  395
 


In [3]:
##########
np.random.seed(42)
n = X.shape[0]
indices = np.random.permutation(n)
split = int(0.9 * n)
lhs_idx = indices[:split]
rhs_idx = indices[split:]

lhs_idx_negative = lhs_idx[y[lhs_idx] == -1]
X_LHS_random = X[lhs_idx_negative]
X_RHS_random = X[rhs_idx]
y_RHS_random = y[rhs_idx]

xxneg = y[lhs_idx]

len(lhs_idx_negative), len(lhs_idx), len(rhs_idx), np.sum(xxneg == -1), np.sum(xxneg == 1)


(206, 355, 40, 206, 149)

## Run the coverage radius algorithm

In [4]:
pos_mask   = (y_RHS_random == 1)
X_RHS_pos  = X_RHS_random[pos_mask]      
pos_labels = y_RHS_random[pos_mask]     

print(f"Positive RHS nodes: {len(X_RHS_pos)}")
 
            
results = []

for R_budget in [4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]:
    
    radii, covered = radius_greedy(X_LHS_random, X_RHS_pos, R_budget)
    
    print(f"\nBudget used: {R_budget}")
    print(f"Agents covered: {covered.sum()} out of {len(X_LHS_random)}")
    print("Non-zero radii for positive targets:")
    for i, r in enumerate(radii):
        if r > 0:
            print(f"  Target {i:2d}: radius {r:.3f}")
    
    row = {
        "Budget": R_budget,
        "Agents_Covered": covered.sum(),
        "Total_Agents": len(X_LHS_random)
    }

    for i, r in enumerate(radii):
        if r > 0:
            row[f"Target_{i}_Radius"] = r
    
    results.append(row)

datasf = pd.DataFrame(results)
datasf.to_csv("./cov_results/math_rapproach.csv", index=False)


Positive RHS nodes: 17

Budget used: 4
Agents covered: 4 out of 206
Non-zero radii for positive targets:
  Target 15: radius 3.959

Budget used: 4.5
Agents covered: 7 out of 206
Non-zero radii for positive targets:
  Target 15: radius 4.197

Budget used: 5
Agents covered: 17 out of 206
Non-zero radii for positive targets:
  Target 15: radius 4.991

Budget used: 5.5
Agents covered: 38 out of 206
Non-zero radii for positive targets:
  Target 15: radius 5.487

Budget used: 6
Agents covered: 58 out of 206
Non-zero radii for positive targets:
  Target 15: radius 5.999

Budget used: 6.5
Agents covered: 92 out of 206
Non-zero radii for positive targets:
  Target 15: radius 6.490

Budget used: 7
Agents covered: 121 out of 206
Non-zero radii for positive targets:
  Target 15: radius 6.991

Budget used: 7.5
Agents covered: 146 out of 206
Non-zero radii for positive targets:
  Target 15: radius 7.459

Budget used: 8
Agents covered: 171 out of 206
Non-zero radii for positive targets:
  Target 15: 